# Microsoft Agent Framework Demo
Please visit https://aka.ms/agentframework for the official code samples

In [17]:
from agent_framework import ChatAgent, AgentProtocol, AgentThread, HostedMCPTool
from agent_framework.azure import AzureAIAgentClient
from azure.identity.aio import AzureCliCredential
from typing import Any

In [18]:
async def main():
    async with (
        AzureCliCredential() as credential,
        ChatAgent(
            chat_client=AzureAIAgentClient(async_credential=credential),
            instructions="You are good at telling jokes."
        ) as agent,
    ):
        result = await agent.run("Tell me a joke about a pirate.")
        print(result.text)

await main()

Why did the pirate go to school?

Because he wanted to improve his "arrrrr-ticulation"!


In [19]:
"""
Azure AI Agent with Hosted MCP Example

This sample demonstrates integration of Azure AI Agents with hosted Model Context Protocol (MCP)
servers, including user approval workflows for function call security.
"""

async def handle_approvals_with_thread(query: str, agent: "AgentProtocol", thread: "AgentThread"):
    """Here we let the thread deal with the previous responses, and we just rerun with the approval."""
    from agent_framework import ChatMessage

    result = await agent.run(query, thread=thread, store=True)
    while len(result.user_input_requests) > 0:
        new_input: list[Any] = []
        for user_input_needed in result.user_input_requests:
            print(
                f"User Input Request for function from {agent.name}: {user_input_needed.function_call.name}"
                f" with arguments: {user_input_needed.function_call.arguments}"
            )
            new_input.append(
                ChatMessage(
                    role="user",
                    contents=[user_input_needed.create_response(True)],
                ),
            )
        result = await agent.run(new_input, thread=thread, store=True)
    return result


async def main() -> None:
    """Example showing Hosted MCP tools for a Azure AI Agent."""
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as chat_client,
    ):
        # enable azure-ai observability
        await chat_client.setup_azure_ai_observability()
        agent = chat_client.create_agent(
            name="DocsAgent",
            instructions="You are a helpful assistant that can help with microsoft documentation questions.",
            tools=HostedMCPTool(
                name="Microsoft Learn MCP",
                url="https://learn.microsoft.com/api/mcp",
            ),
        )
        thread = agent.get_new_thread()
        # First query
        query1 = "How to create an Azure storage account using az cli?"
        print(f"User: {query1}")
        result1 = await handle_approvals_with_thread(query1, agent, thread)
        print(f"{agent.name}: {result1}\n")
        print("\n=======================================\n")
        # Second query
        query2 = "What is Microsoft Semantic Kernel?"
        print(f"User: {query2}")
        result2 = await handle_approvals_with_thread(query2, agent, thread)
        print(f"{agent.name}: {result2}\n")

await main()

[2025-10-01 16:38:18 - c:\Users\gugregor\AppData\Local\Programs\Python\Python313\Lib\site-packages\agent_framework_azure_ai\_chat_client.py:223 - WARNING] No Application Insights connection string found for the Azure AI Project, please call setup_observability() manually.


User: How to create an Azure storage account using az cli?
User Input Request for function from DocsAgent: microsoft_code_sample_search with arguments: {"query":"create Azure storage account using az cli","language":"azurecli"}
User Input Request for function from DocsAgent: microsoft_code_sample_search with arguments: {"query":"create Azure storage account using az cli","language":"azurecli"}
DocsAgent: To create an Azure Storage Account using Azure CLI, use the following command:

```sh
az storage account create \
  --name <storage-account-name> \
  --resource-group <resource-group-name> \
  --location <location> \
  --sku Standard_LRS \
  --kind StorageV2
```

**Parameters:**
- `<storage-account-name>`: Must be unique across Azure.
- `<resource-group-name>`: The resource group where the storage account will be created.
- `<location>`: The Azure region, e.g., `eastus`, `westus`.
- `--sku`: The SKU type, e.g., `Standard_LRS`.
- `--kind`: Storage account type, e.g., `StorageV2`.

**Exa